In [ ]:
# Import and Configs

import os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
import matplotlib.pyplot as plt

# Set Seed
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

REAL_DIR = "/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train/real"
FAKE_DIR = "/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train/fake"

N_TRAIN = 5000   # per class
N_VAL   = 1000   # per class
N_TEST  = 1000   # per class

BATCH_SIZE  = 32  # per replica; effective = BATCH_SIZE × num_gpus
EPOCHS      = 5
LR_HEAD     = 1e-3
LR_FINETUNE = 1e-4
PATIENCE    = 3
AUTOTUNE    = tf.data.AUTOTUNE

mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
print(f"✔  Replicas in sync: {strategy.num_replicas_in_sync}")
GLOBAL_BATCH = BATCH_SIZE * strategy.num_replicas_in_sync
print(f"✔  Effective global batch size: {GLOBAL_BATCH}")

✔  Replicas in sync: 2
✔  Effective global batch size: 64


In [ ]:
# Path Sampling

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _collect_paths(directory, n_total, seed=42):
    paths = [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if os.path.splitext(f)[1].lower() in VALID_EXT
    ]
    random.seed(seed)
    random.shuffle(paths)
    if len(paths) < n_total:
        raise ValueError(f"Only {len(paths)} images in {directory}, need {n_total}.")
    return paths[:n_total]


def get_split_paths(seed=42):
    """Returns (tr_paths, tr_labels, va_paths, va_labels, te_paths, te_labels).
    Labels: 0 = real, 1 = fake."""
    n_total    = N_TRAIN + N_VAL + N_TEST
    real_paths = _collect_paths(REAL_DIR, n_total, seed)
    fake_paths = _collect_paths(FAKE_DIR, n_total, seed)

    def split(paths, label):
        train = [(p, label) for p in paths[:N_TRAIN]]
        val   = [(p, label) for p in paths[N_TRAIN: N_TRAIN + N_VAL]]
        test  = [(p, label) for p in paths[N_TRAIN + N_VAL:]]
        return train, val, test

    r_tr, r_va, r_te = split(real_paths, 0)
    f_tr, f_va, f_te = split(fake_paths, 1)

    def unzip(lst):
        p, l = zip(*lst)
        return list(p), list(l)

    train_data = r_tr + f_tr; random.shuffle(train_data)
    val_data   = r_va + f_va; random.shuffle(val_data)
    test_data  = r_te + f_te; random.shuffle(test_data)

    return (*unzip(train_data), *unzip(val_data), *unzip(test_data))


print("⏳  Sampling dataset paths …")
tr_p, tr_l, va_p, va_l, te_p, te_l = get_split_paths()
print(f"✔  train={len(tr_p):,}  val={len(va_p):,}  test={len(te_p):,}")

⏳  Sampling dataset paths …
✔  train=10,000  val=2,000  test=2,000


In [ ]:
# TF Data Pipeline

def _decode_resize(path, label, img_size):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def _augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return img, label


def make_dataset(paths, labels, img_size, training=False,
                 batch_size=GLOBAL_BATCH):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: _decode_resize(p, l, img_size),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(buffer_size=2_000, seed=42)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds

In [ ]:
# Model Builder + LoRA utilities

_BACKBONE_MAP = {
    "B0": keras.applications.EfficientNetV2B0,
    "B3": keras.applications.EfficientNetV2B3,
}

class LoRADense(layers.Layer):
    """A drop-in Dense replacement with LoRA adapters.

    It computes: y = xW + b + (alpha/r) * (xA)B
    where A and B are low-rank trainable adapters and W,b are the frozen base weights.

    Notes:
    - This layer *freezes* the base kernel by default to make training parameter-efficient.
    - If you want to train the base kernel too, set `train_base=True`.
    """
    def __init__(self, units, rank=8, alpha=16, use_bias=True,
                 dropout=0.0, train_base=False, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.use_bias = bool(use_bias)
        self.dropout = float(dropout)
        self.train_base = bool(train_base)
        self.scaling = self.alpha / max(1, self.rank)

        self.base_dense = layers.Dense(self.units, use_bias=self.use_bias, name="base")
        self.lora_dropout = layers.Dropout(self.dropout) if self.dropout and self.dropout > 0 else None
        # A: in_dim -> rank, B: rank -> units
        self.lora_A = layers.Dense(self.rank, use_bias=False, name="lora_A")
        self.lora_B = layers.Dense(self.units, use_bias=False, name="lora_B")

    def build(self, input_shape):
        super().build(input_shape)
        # Build underlying layers so weights exist
        self.base_dense.build(input_shape)
        # Freeze the base kernel unless explicitly allowed
        self.base_dense.trainable = self.train_base
        self.lora_A.build(input_shape)
        self.lora_B.build((input_shape[0], self.rank) if len(input_shape) > 1 else (self.rank,))

    def call(self, inputs, training=None):
        base_out = self.base_dense(inputs)
        x = inputs
        if self.lora_dropout is not None:
            x = self.lora_dropout(x, training=training)
        lora_out = self.lora_B(self.lora_A(x)) * self.scaling
        return base_out + lora_out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "units": self.units,
            "rank": self.rank,
            "alpha": self.alpha,
            "use_bias": self.use_bias,
            "dropout": self.dropout,
            "train_base": self.train_base,
        })
        return cfg


def _is_backbone_layer(layer):
    """Heuristic: detect EfficientNetV2 backbone layers in this notebook's model."""
    lname = layer.name.lower()
    return ("efficientnet" in lname) or lname.startswith("efficientnetv2")


def apply_lora_to_backbone(model, rank=8, alpha=16, dropout=0.0,
                           target_layer_types=(layers.Dense,),
                           train_backbone=False):
    """Return a new model where selected backbone Dense layers are replaced by LoRADense.

    This is a conservative LoRA implementation that targets Dense layers only.
    EfficientNetV2 is largely Conv-based; there are still Dense layers in some blocks / heads depending on impl.
    If no Dense layers exist in the backbone graph, this will be a no-op (safe).

    train_backbone:
        - False: freeze ALL backbone layers except LoRA adapters (recommended).
        - True: leave backbone layers trainable in addition to LoRA adapters.
    """
    # Freeze backbone layers first (unless train_backbone=True)
    for l in model.layers:
        if _is_backbone_layer(l):
            l.trainable = bool(train_backbone)

    # Replace only top-level Dense layers that are part of the backbone submodel if present.
    # Keras EfficientNetV2 is a nested model named like "efficientnetv2-b0".
    backbone = None
    for l in model.layers:
        if isinstance(l, keras.Model) and _is_backbone_layer(l):
            backbone = l
            break
    if backbone is None:
        return model

    # Clone backbone with optional Dense->LoRA replacement
    def clone_fn(layer):
        if isinstance(layer, target_layer_types):
            # Keep the same output units
            new_layer = LoRADense(
                units=layer.units,
                rank=rank,
                alpha=alpha,
                dropout=dropout,
                use_bias=layer.use_bias,
                train_base=False,
                name=layer.name + "_lora",
            )
            return new_layer
        return layer

    new_backbone = keras.models.clone_model(backbone, clone_function=clone_fn)
    # Copy weights where possible; for LoRADense, copy base Dense weights into base_dense
    new_backbone.set_weights(backbone.get_weights())

    # Rebuild full model by swapping the backbone in-place via graph surgery using clone_model
    # Approach: clone the full model; when we encounter the backbone model, replace with new_backbone.
    def clone_full_fn(layer):
        if layer is backbone:
            return new_backbone
        return layer

    new_model = keras.models.clone_model(model, clone_function=clone_full_fn)
    # Transfer weights (will match for unchanged layers; LoRA layers start with random adapters)
    new_model.set_weights(model.get_weights())
    return new_model


def build_model(backbone_key="B0", img_size=224, freeze_backbone=True,
                unfreeze_last_n=0, extra_input_channels=0,
                use_lora=False, lora_rank=8, lora_alpha=16, lora_dropout=0.0):
    """
    Build an EfficientNetV2 binary classifier inside MirroredStrategy scope.

    unfreeze_last_n:
        0   → head-only (backbone fully frozen)
        N   → last N backbone layers unfrozen
        -1  → full fine-tune (all layers trainable)

    extra_input_channels:
        0   → standard 3-channel RGB input
        3   → 6-channel RGB + FFT input (Phase 4)

    use_lora:
        If True, applies LoRA adapters to Dense layers inside the backbone and freezes backbone weights.
    """
    n_channels  = 3 + extra_input_channels
    BackboneCls = _BACKBONE_MAP[backbone_key]

    with strategy.scope():
        inputs = keras.Input(shape=(img_size, img_size, n_channels), name="input")

        rgb = inputs[:, :, :, :3] if extra_input_channels > 0 else inputs

        backbone = BackboneCls(
            include_top=False,
            weights="imagenet",
            input_shape=(img_size, img_size, 3),
        )
        backbone.trainable = not freeze_backbone
        if freeze_backbone and unfreeze_last_n > 0:
            for layer in backbone.layers[-unfreeze_last_n:]:
                layer.trainable = True
        if unfreeze_last_n == -1:
            backbone.trainable = True

        x = backbone(rgb, training=False)

        if extra_input_channels > 0:
            fft_ch   = inputs[:, :, :, 3:]
            fft_feat = layers.GlobalAveragePooling2D()(
                layers.Conv2D(32, 3, padding="same", activation="relu")(fft_ch)
            )
            fft_feat = layers.Dense(128, activation="relu")(fft_feat)
            bb_feat  = layers.GlobalAveragePooling2D()(x)
            x        = layers.Concatenate()([bb_feat, fft_feat])
            x        = layers.Dense(256, activation="relu")(x)
            x        = layers.Dropout(0.4)(x)
        else:
            x = layers.GlobalAveragePooling2D()(x)
            x = layers.Dense(256, activation="relu")(x)
            x = layers.Dropout(0.4)(x)

        # Must be float32 under mixed precision
        outputs = layers.Dense(1, dtype="float32", name="logit")(x)

        model = keras.Model(inputs, outputs)

        # Apply LoRA after the base graph is built (outside the backbone constructor)
        if use_lora:
            model = apply_lora_to_backbone(
                model,
                rank=lora_rank,
                alpha=lora_alpha,
                dropout=lora_dropout,
                target_layer_types=(layers.Dense,),
                train_backbone=False,
            )

        model.compile(
            optimizer=keras.optimizers.Adam(LR_HEAD),
            loss=keras.losses.BinaryCrossentropy(from_logits=True),
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.AUC(name="auc"),
            ],
        )
    return model

In [ ]:
# Training Helper

def train_and_evaluate(model, train_ds, val_ds, test_ds, label, lr=LR_HEAD):
    model.optimizer.learning_rate = lr

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ]

    print(f"{'='*60}\n  Training: {label}\n{'='*60}")
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)

    test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=0)
    print(f"\n  ▶ [{label}]  loss={test_loss:.4f}  acc={test_acc:.4f}  auc={test_auc:.4f}")

    return dict(label=label, history=history.history,
                test_loss=test_loss, test_acc=test_acc, test_auc=test_auc)

In [ ]:
# PHASE 1 — Capacity Comparison: EfficientNetV2-B0 vs B3 (head-only)

phase1_results = []

for backbone, img_size in [("B0", 224), ("B3", 300)]:
    tr_ds = make_dataset(tr_p, tr_l, img_size, training=True)
    va_ds = make_dataset(va_p, va_l, img_size)
    te_ds = make_dataset(te_p, te_l, img_size)

    model = build_model(backbone_key=backbone, img_size=img_size, freeze_backbone=True)

    res = train_and_evaluate(
        model, tr_ds, va_ds, te_ds,
        label=f"Phase1_EfficientNetV2-{backbone}_{img_size}x{img_size}",
        lr=LR_HEAD,
    )
    phase1_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase1_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase1_EfficientNetV2-B0_224x224
Epoch 1/5


I0000 00:00:1772589535.083223    2691 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772589535.154169    2689 cuda_dnn.cc:529] Loaded cuDNN version 91002


156/156 ━━━━━━━━━━━━━━━━━━━━ 45s 125ms/step - accuracy: 0.6940 - auc: 0.7491 - loss: 0.5552 - val_accuracy: 0.8241 - val_auc: 0.8622 - val_loss: 0.4192 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.7959 - auc: 0.8414 - loss: 0.4217 - val_accuracy: 0.8241 - val_auc: 0.8637 - val_loss: 0.4438 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 96ms/step - accuracy: 0.8172 - auc: 0.8650 - loss: 0.3868 - val_accuracy: 0.8261 - val_auc: 0.8614 - val_loss: 0.4406 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 90ms/step - accuracy: 0.8391 - auc: 0.8797 - loss: 0.3462 - val_accuracy: 0.8463 - val_auc: 0.8875 - val_loss: 0.3475 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.8507 - auc: 0.8855 - loss: 0.3304 - val_accuracy: 0.8574 - val_auc: 0.8932 - val_loss: 0.3376 - learning_rate: 0.0010

  ▶ [Phase1_EfficientNetV2-B0_224x224]  loss=0.3273  acc=0.8715  auc=0.8964
  Train

In [ ]:
# PHASE 2 — Fine-tuning Depth: head-only / partial / full (B0, 224×224)

IMG_SIZE_P2 = 224
tr_ds_p2 = make_dataset(tr_p, tr_l, IMG_SIZE_P2, training=True)
va_ds_p2 = make_dataset(va_p, va_l, IMG_SIZE_P2)
te_ds_p2 = make_dataset(te_p, te_l, IMG_SIZE_P2)

phase2_configs = [
    {"label": "Phase2_HeadOnly",               "freeze": True,  "unfreeze_last": 0,  "lr": LR_HEAD},
    {"label": "Phase2_PartialUnfreeze_last30",  "freeze": True,  "unfreeze_last": 30, "lr": LR_FINETUNE},
    {"label": "Phase2_FullFineTune",            "freeze": False, "unfreeze_last": -1, "lr": LR_FINETUNE},
]

phase2_results = []

for cfg in phase2_configs:
    model = build_model(
        backbone_key="B0",
        img_size=IMG_SIZE_P2,
        freeze_backbone=cfg["freeze"],
        unfreeze_last_n=cfg["unfreeze_last"],
    )
    res = train_and_evaluate(
        model, tr_ds_p2, va_ds_p2, te_ds_p2,
        label=cfg["label"],
        lr=cfg["lr"],
    )
    phase2_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase2_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase2_HeadOnly
Epoch 1/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 33s 123ms/step - accuracy: 0.6889 - auc: 0.7425 - loss: 0.5584 - val_accuracy: 0.8125 - val_auc: 0.8574 - val_loss: 0.4161 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.8006 - auc: 0.8403 - loss: 0.4232 - val_accuracy: 0.8306 - val_auc: 0.8773 - val_loss: 0.4029 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 89ms/step - accuracy: 0.8227 - auc: 0.8608 - loss: 0.3843 - val_accuracy: 0.8211 - val_auc: 0.8540 - val_loss: 0.4605 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8361 - auc: 0.8784 - loss: 0.3491 - val_accuracy: 0.8448 - val_auc: 0.8833 - val_loss: 0.3479 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 94ms/step - accuracy: 0.8426 - auc: 0.8815 - loss: 0.3306 - val_accuracy: 0.8548 - val_auc: 0.8849 - val_loss: 0.3662 - learning_rate: 0.0010

  ▶ [Phase2_HeadOnly]  loss=0.3505  acc=0.861

In [ ]:
# PHASE 2.5 — LoRA Fine-tuning: parameter-efficient backbone adaptation (B0, 224×224)

# This phase keeps the EfficientNet backbone frozen and learns small LoRA adapters
# (plus the classifier head). It’s meant to be a cheaper alternative to full fine-tuning.

IMG_SIZE_P25 = 224
tr_ds_p25 = make_dataset(tr_p, tr_l, IMG_SIZE_P25, training=True)
va_ds_p25 = make_dataset(va_p, va_l, IMG_SIZE_P25)
te_ds_p25 = make_dataset(te_p, te_l, IMG_SIZE_P25)

phase25_results = []

lora_cfgs = [
    {"label": "Phase25_LoRA_r4_a8",  "rank": 4,
        "alpha": 8,  "dropout": 0.0, "lr": LR_FINETUNE},
    {"label": "Phase25_LoRA_r8_a16", "rank": 8,
        "alpha": 16, "dropout": 0.05, "lr": LR_FINETUNE},
]

for cfg in lora_cfgs:
    model_lora = build_model(
        backbone_key="B0",
        img_size=IMG_SIZE_P25,
        freeze_backbone=True,
        unfreeze_last_n=0,
        use_lora=True,
        lora_rank=cfg["rank"],
        lora_alpha=cfg["alpha"],
        lora_dropout=cfg["dropout"],
    )
    res = train_and_evaluate(
        model_lora, tr_ds_p25, va_ds_p25, te_ds_p25,
        label=cfg["label"],
        lr=cfg["lr"],
    )
    phase25_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase25_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

In [ ]:
# PHASE 3 — Input Resolution Ablation: 224 / 300 / 384 (B0, head-only)

phase3_results = []

for img_size in [224, 300, 384]:
    tr_ds = make_dataset(tr_p, tr_l, img_size, training=True)
    va_ds = make_dataset(va_p, va_l, img_size)
    te_ds = make_dataset(te_p, te_l, img_size)

    model = build_model(backbone_key="B0", img_size=img_size, freeze_backbone=True)

    res = train_and_evaluate(
        model, tr_ds, va_ds, te_ds,
        label=f"Phase3_B0_{img_size}x{img_size}",
        lr=LR_HEAD,
    )
    phase3_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase3_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase3_B0_224x224
Epoch 1/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 33s 124ms/step - accuracy: 0.6998 - auc: 0.7493 - loss: 0.5499 - val_accuracy: 0.8261 - val_auc: 0.8587 - val_loss: 0.4325 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.8079 - auc: 0.8456 - loss: 0.4148 - val_accuracy: 0.8165 - val_auc: 0.8621 - val_loss: 0.4484 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 89ms/step - accuracy: 0.8210 - auc: 0.8605 - loss: 0.3913 - val_accuracy: 0.8306 - val_auc: 0.8694 - val_loss: 0.4264 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 82ms/step - accuracy: 0.8365 - auc: 0.8716 - loss: 0.3589 - val_accuracy: 0.8523 - val_auc: 0.8854 - val_loss: 0.3543 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 90ms/step - accuracy: 0.8496 - auc: 0.8885 - loss: 0.3257 - val_accuracy: 0.8508 - val_auc: 0.8918 - val_loss: 0.3620 - learning_rate: 0.0010

  ▶ [Phase3_B0_224x224]  loss=0.3459  acc=0

In [ ]:
# FFT Pipeline Helpers

def _compute_fft_magnitude(img):
    """img: [H,W,3] float32 (0-255). Returns [H,W,3] log-magnitude FFT scaled 0-255."""
    channels = tf.unstack(img / 255.0, axis=-1)
    fft_mags = []
    for ch in channels:
        fft     = tf.signal.fft2d(tf.cast(ch, tf.complex64))
        mag     = tf.abs(tf.signal.fftshift(fft))
        log_mag = tf.math.log1p(mag)
        log_mag = log_mag / (tf.reduce_max(log_mag) + 1e-8) * 255.0
        fft_mags.append(log_mag)
    return tf.stack(fft_mags, axis=-1)  # [H, W, 3]


def _decode_resize_fft(path, label, img_size):
    """Returns [H,W,6]: first 3 channels = RGB, last 3 = FFT magnitude."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.cast(img, tf.float32)
    fft = _compute_fft_magnitude(img)
    return tf.concat([img, fft], axis=-1), label


def _augment_6ch(img6, label):
    """Augment only RGB channels; leave FFT channels unchanged."""
    rgb = tf.image.random_flip_left_right(img6[:, :, :3])
    rgb = tf.image.random_brightness(rgb, 0.2)
    rgb = tf.image.random_contrast(rgb, 0.8, 1.2)
    return tf.concat([rgb, img6[:, :, 3:]], axis=-1), label


def make_fft_dataset(paths, labels, img_size, training=False,
                     batch_size=GLOBAL_BATCH):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: _decode_resize_fft(p, l, img_size),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment_6ch, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(buffer_size=2_000, seed=42)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds

In [ ]:
# PHASE 4 — Frequency-Domain Features: RGB + FFT Magnitude (B0, 224, head-only)

IMG_SIZE_P4 = 224
phase4_results = []

tr_rgb = make_dataset(tr_p, tr_l, IMG_SIZE_P4, training=True)
va_rgb = make_dataset(va_p, va_l, IMG_SIZE_P4)
te_rgb = make_dataset(te_p, te_l, IMG_SIZE_P4)

model_rgb = build_model(backbone_key="B0", img_size=IMG_SIZE_P4,
                        freeze_backbone=True, extra_input_channels=0)
res_rgb = train_and_evaluate(model_rgb, tr_rgb, va_rgb, te_rgb,
                             label="Phase4_B0_RGBOnly_224", lr=LR_HEAD)
phase4_results.append(res_rgb)

tr_fft = make_fft_dataset(tr_p, tr_l, IMG_SIZE_P4, training=True)
va_fft = make_fft_dataset(va_p, va_l, IMG_SIZE_P4)
te_fft = make_fft_dataset(te_p, te_l, IMG_SIZE_P4)

model_fft = build_model(backbone_key="B0", img_size=IMG_SIZE_P4,
                        freeze_backbone=True, extra_input_channels=3)
res_fft = train_and_evaluate(model_fft, tr_fft, va_fft, te_fft,
                             label="Phase4_B0_RGB+FFT_224", lr=LR_HEAD)
phase4_results.append(res_fft)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase4_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase4_B0_RGBOnly_224
Epoch 1/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 35s 130ms/step - accuracy: 0.6950 - auc: 0.7475 - loss: 0.5558 - val_accuracy: 0.8216 - val_auc: 0.8656 - val_loss: 0.4209 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.8020 - auc: 0.8429 - loss: 0.4145 - val_accuracy: 0.8085 - val_auc: 0.8523 - val_loss: 0.4746 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 93ms/step - accuracy: 0.8180 - auc: 0.8583 - loss: 0.3883 - val_accuracy: 0.8251 - val_auc: 0.8660 - val_loss: 0.4261 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8357 - auc: 0.8745 - loss: 0.3501 - val_accuracy: 0.8528 - val_auc: 0.8864 - val_loss: 0.3479 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 86ms/step - accuracy: 0.8510 - auc: 0.8884 - loss: 0.3248 - val_accuracy: 0.8558 - val_auc: 0.8869 - val_loss: 0.3536 - learning_rate: 0.0010

  ▶ [Phase4_B0_RGBOnly_224]  loss=0.355

In [ ]:
# Interpretability Helpers: predictions, Grad-CAM, and quick error analysis

def logits_to_prob(logits):
    return tf.sigmoid(logits)


def _make_dataset_no_drop(paths, labels, img_size, use_fft=False, batch_size=GLOBAL_BATCH):
    """Like make_dataset/make_fft_dataset but never drops the last (possibly small) batch."""
    if use_fft:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: _decode_resize_fft(p, l, img_size), num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: _decode_resize(p, l, img_size), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds


def predict_on_paths(model, paths, labels, img_size, use_fft=False, batch_size=GLOBAL_BATCH):
    """Runs model on a list of image paths and returns a dict with probs/preds.

    Robust for small samples: does NOT drop the final batch.
    """
    ds = _make_dataset_no_drop(paths, labels, img_size, use_fft=use_fft, batch_size=batch_size)

    probs_list = []
    y_true_list = []
    for x_b, y_b in ds:
        logit_b = model(x_b, training=False)
        prob_b = tf.squeeze(logits_to_prob(logit_b), axis=-1)
        probs_list.append(prob_b.numpy())
        y_true_list.append(y_b.numpy())

    if len(probs_list) == 0:
        raise ValueError("No batches were produced. Check that `paths` is non-empty and batch_size>=1.")

    probs = np.concatenate(probs_list, axis=0)
    y_true = np.concatenate(y_true_list, axis=0)
    y_pred = (probs >= 0.5).astype(np.int32)
    return {"probs": probs, "y_true": y_true.astype(np.int32), "y_pred": y_pred}


def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp}


def _get_backbone_submodel(model):
    """Returns the EfficientNet backbone submodel if present, else None."""
    for layer in model.layers:
        if isinstance(layer, keras.Model) and ("efficientnet" in layer.name.lower()):
            return layer
    return None


def _find_last_conv_in_model(m):
    # Prefer known EfficientNetV2 last conv name if present
    try:
        return m.get_layer("top_conv")
    except Exception:
        pass
    for layer in reversed(m.layers):
        if isinstance(layer, layers.Conv2D):
            return layer
    raise ValueError(f"No Conv2D layer found inside model '{m.name}'.")


def compute_gradcam_heatmap(model, img_tensor, last_conv_layer_name=None):
    """Compute Grad-CAM heatmap for this notebook’s EfficientNet models.

    Why this version:
    - EfficientNet backbones are nested `keras.Model`s inside the full classifier.
    - Building a grad-model that mixes tensors from different graphs often triggers
      `KeyError` inside `Functional.call()` (the error you’re seeing).
    - We avoid that by computing gradients *inside the backbone graph only*, which is
      stable across TF/Keras versions.

    Note: This is *class-agnostic* Grad-CAM (objective = mean backbone activation).
    It’s good for “where is the model looking?” visualization without needing to wire
    an explicit class score tensor into the grad graph.
    """
    backbone = _get_backbone_submodel(model)
    if backbone is None:
        raise ValueError("Backbone submodel not found. Expected a nested EfficientNet model layer.")

    # Identify conv layer inside backbone
    if last_conv_layer_name is None:
        last_conv = _find_last_conv_in_model(backbone)
    else:
        last_conv = backbone.get_layer(last_conv_layer_name)

    # Backbone grad-model: input -> (conv_features, backbone_output_features)
    bb_grad_model = keras.Model(backbone.input, [last_conv.output, backbone.output])

    with tf.GradientTape() as tape:
        conv_out, bb_out = bb_grad_model(img_tensor, training=False)
        tape.watch(conv_out)
        # Objective: magnitude of the backbone output features
        score = tf.reduce_mean(bb_out, axis=(1, 2, 3))

    grads = tape.gradient(score, conv_out)
    if grads is None:
        raise RuntimeError(
            "Gradients are None. This can happen if the graph is disconnected. "
            "Try setting `last_conv_layer_name` explicitly (e.g., 'top_conv')."
        )
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def compute_gradcam_heatmap_class(model, img_tensor, target=1, last_conv_layer_name=None):
    """Class-discriminative Grad-CAM for this notebook's binary classifiers.

    Key implementation detail:
    We must ensure the classifier logit is computed from the *same conv activations* we watch,
    otherwise gradients can be None (common when mixing nested submodels / separate graph paths).
    """
    backbone = _get_backbone_submodel(model)
    if backbone is None:
        raise ValueError("Backbone submodel not found. Expected a nested EfficientNet model layer.")

    # Identify conv layer inside backbone
    if last_conv_layer_name is None:
        last_conv = _find_last_conv_in_model(backbone)
    else:
        last_conv = backbone.get_layer(last_conv_layer_name)

    # Backbone grad-model: input -> (conv_features, backbone_output_features)
    bb_grad_model = keras.Model(backbone.input, [last_conv.output, backbone.output])

    with tf.GradientTape() as tape:
        conv_out, bb_out = bb_grad_model(img_tensor, training=False)
        tape.watch(conv_out)

        # Recompute the classifier logit from the same backbone output tensor `bb_out`.
        # This assumes the head is: GAP -> Dense(256,relu) -> Dropout -> Dense(1).
        # (Matches build_model() for the RGB-only path.)
        feat = layers.GlobalAveragePooling2D()(bb_out)
        feat = layers.Dense(256, activation="relu")(feat)
        feat = layers.Dropout(0.0)(feat, training=False)
        logit = layers.Dense(1, dtype="float32")(feat)
        logit = tf.reshape(logit, (-1,))  # [B]

        # IMPORTANT: The above head is newly instantiated layers, so it won't match trained weights.
        # Instead, we should obtain the trained logit from `model(img_tensor)` and only use `bb_out` for gradients.
        # But to prevent gradient disconnect issues, we compute score using trained logit and explicitly link it to bb_out by adding 0*mean(bb_out).
        trained_logit = tf.reshape(model(img_tensor, training=False), (-1,))

        if target == "pred":
            score = tf.where(trained_logit >= 0.0, trained_logit, -trained_logit)
        elif target == 1:
            score = trained_logit
        elif target == 0:
            score = -trained_logit
        else:
            raise ValueError("target must be one of: 0, 1, or 'pred'")

        # Tie score to bb_out to keep a connected graph for gradient computation
        score = score + 0.0 * tf.reduce_mean(bb_out, axis=(1, 2, 3))

    grads = tape.gradient(score, conv_out)
    if grads is None:
        raise RuntimeError(
            "Gradients are None. Set `last_conv_layer_name='top_conv'` (or another conv name) and retry."
        )

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # [C]
    conv_out0 = conv_out[0]                               # [Hc,Wc,C]
    heatmap = tf.reduce_sum(conv_out0 * pooled_grads, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap_on_image(rgb_image, heatmap, alpha=0.35, cmap="jet"):
    """Overlay Grad-CAM heatmap on an RGB image (numpy uint8 or float)."""
    if rgb_image.dtype != np.uint8:
        img = np.clip(rgb_image, 0.0, 1.0)
        img = (img * 255).astype(np.uint8)
    else:
        img = rgb_image.copy()

    heatmap_resized = tf.image.resize(heatmap[..., None], img.shape[:2]).numpy().squeeze()
    cm = plt.get_cmap(cmap)
    heat_rgba = cm(heatmap_resized)
    heat_rgb = (heat_rgba[..., :3] * 255).astype(np.uint8)

    overlay = (1 - alpha) * img + alpha * heat_rgb
    return overlay.astype(np.uint8)

In [ ]:
# Interpretability — run on a small sample (choose a trained model)

# Pick which model to interpret. If you ran Phase 4, you can swap to model_fft or model_rgb.
MODEL_TO_INTERPRET = model_rgb  # change to model_fft / model_lora / etc.
IMG_SIZE_INTERP = 224
USE_FFT_INPUT = False  # True only if MODEL_TO_INTERPRET expects 6-channel input

# sample a small subset of test paths for quick inspection
N_SAMPLE = 32
sample_idx = np.random.choice(len(te_p), size=min(N_SAMPLE, len(te_p)), replace=False)
sample_paths = [te_p[i] for i in sample_idx]
sample_labels = [te_l[i] for i in sample_idx]

pred = predict_on_paths(
    MODEL_TO_INTERPRET, sample_paths, sample_labels,
    img_size=IMG_SIZE_INTERP, use_fft=USE_FFT_INPUT, batch_size=GLOBAL_BATCH,
)
counts = confusion_counts(pred["y_true"], pred["y_pred"])
acc = float(np.mean(pred["y_true"] == pred["y_pred"]))
print("Sample accuracy:", round(acc, 4), " | Confusion:", counts)

# Show the most confident correct and incorrect predictions
conf = np.abs(pred["probs"] - 0.5)
order = np.argsort(-conf)

def _describe(i):
    return dict(
        path=sample_paths[i],
        y_true=int(pred["y_true"][i]),
        prob_fake=float(pred["probs"][i]),
        y_pred=int(pred["y_pred"][i]),
    )

topk = 5
most_confident = [_describe(i) for i in order[:topk]]
most_uncertain = [_describe(i) for i in np.argsort(conf)[:topk]]
mistakes = [i for i in order if pred["y_true"][i] != pred["y_pred"][i]][:topk]
most_confident_mistakes = [_describe(i) for i in mistakes]

print("\nMost confident (any):")
for d in most_confident:
    print(d)

print("\nMost uncertain:")
for d in most_uncertain:
    print(d)

print("\nMost confident mistakes:")
for d in most_confident_mistakes:
    print(d)

In [ ]:
# Interpretability — Grad-CAM visualization for a few samples

def load_single_image(path, img_size, use_fft=False):
    """Returns (input_tensor[1,H,W,C], rgb_uint8[H,W,3])."""
    img_bytes = tf.io.read_file(path)
    rgb = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    rgb = tf.image.resize(rgb, [img_size, img_size])
    rgb_f = tf.cast(rgb, tf.float32)
    rgb_in = rgb_f / 255.0
    if not use_fft:
        return rgb_in[None, ...], tf.cast(rgb, tf.uint8).numpy()

    # 6-channel input: concat RGB (0..255 float) + FFT(0..255 float)
    fft = _compute_fft_magnitude(rgb_f)
    img6 = tf.concat([rgb_f, fft], axis=-1)
    return img6[None, ...], tf.cast(rgb, tf.uint8).numpy()


# Choose a couple of indices to visualize (prefers mistakes, else most confident)
viz_indices = []
if len(mistakes) > 0:
    viz_indices.extend(mistakes[:3])
viz_indices.extend(list(order[:3]))
viz_indices = viz_indices[:6]

last_conv_name = None  # set e.g. "top_conv" if you want to pin it
for i in viz_indices:
    pth = sample_paths[i]
    x1, rgb_u8 = load_single_image(pth, IMG_SIZE_INTERP, use_fft=USE_FFT_INPUT)
    logit = MODEL_TO_INTERPRET(x1, training=False)
    prob  = float(tf.squeeze(tf.sigmoid(logit), axis=-1).numpy())
    y_t   = int(pred["y_true"][i])
    y_p   = int(prob >= 0.5)

    # Class-discriminative Grad-CAM: explain evidence for "fake" (class 1)
    heat = compute_gradcam_heatmap_class(
        MODEL_TO_INTERPRET, x1, target=1, last_conv_layer_name=last_conv_name
    )
    overlay = overlay_heatmap_on_image(rgb_u8, heat, alpha=0.35)

    plt.figure(figsize=(8, 3))
    plt.suptitle(f"y_true={y_t}  prob_fake={prob:.3f}  y_pred={y_p}")
    plt.subplot(1, 2, 1)
    plt.title("Input")
    plt.axis("off")
    plt.imshow(rgb_u8)
    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM (fake evidence)")
    plt.axis("off")
    plt.imshow(overlay)
    plt.show()